# Chapter 4: When Your Coordinates Lie

## Why whitening matters, frame construction, and the metric tensor

Geometric Algebra is built on one crucial assumption: that you are working in an **orthonormal frame**. The basis vectors $\{e_1, e_2, \ldots, e_k\}$ must satisfy:

$$e_i \cdot e_j = \delta_{ij} = \begin{cases} 1 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases}$$

When this holds, inner products measure true angles, bivectors encode true planes, and rotors perform true rotations. Every GA formula you have seen so far — the wedge product, the principal plane decomposition, the rotor-bivector correspondence — *assumes* this orthonormal structure.

But the raw hidden states of a transformer **do not live in an orthonormal frame**. Some dimensions have enormous variance while others are nearly dead. The coordinate axes of $\mathbb{R}^{3584}$ (or whatever the model's hidden dimension is) are not geometrically meaningful — they are artifacts of the weight initialization and training dynamics.

If you apply GA operations directly to raw hidden states, you will get **wrong answers**: dot products will be dominated by high-variance dimensions, bivectors will be tilted toward those dimensions, and the "planes of rotation" you extract will be artifacts of the coordinate system rather than genuine geometric structure.

The fix is **whitening**: constructing a new orthonormal frame in which the covariance is the identity matrix. This chapter shows you why this matters, how it works, and what goes wrong if you skip it.

**By the end of this chapter you will be able to:**
- Explain why raw hidden-state coordinates distort GA operations
- Perform PCA-based whitening and verify orthonormality
- Articulate the role of the metric tensor $g_{ij}$ in Clifford algebra

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga
import layer_time_geometry as core

print("Imports OK")
print(f"NumPy {np.__version__}")

## The Metric Tensor

In a general coordinate system, the inner product between two vectors $\mathbf{u}$ and $\mathbf{v}$ is not simply $\sum_i u_i v_i$. It is:

$$\langle \mathbf{u}, \mathbf{v} \rangle = \sum_{i,j} g_{ij} \, u_i \, v_j = \mathbf{u}^T G \, \mathbf{v}$$

where $G = (g_{ij})$ is the **metric tensor**. In an orthonormal frame, $G = I$ (the identity) and the formula reduces to the ordinary dot product. But in the raw hidden-state coordinates, $G \neq I$ — it equals the **covariance matrix** of the hidden states.

The covariance matrix tells you the "shape" of the data cloud. When it is far from identity:
- High-variance dimensions dominate dot products (they contribute disproportionately).
- Low-variance dimensions are nearly invisible (their contributions are swamped).
- Angles and distances are distorted.

Let us load a model, extract raw hidden states, and see how far the covariance is from identity.

In [ ]:
# ------------------------------------------------------------------
# Load model and extract raw hidden states
# ------------------------------------------------------------------
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")
print(f"Loaded: {model.name}")
print(f"  Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")

# Extract raw hidden states
import torch
H_raw = core.extract_hidden_states(
    model.hf_model, model.tokenizer,
    "Geometric algebra provides a unified language for geometry",
    model.device
)
H_np = H_raw[1:].cpu().numpy()  # skip embedding layer
L, T, p = H_np.shape
print(f"\nRaw hidden states: L={L} layers, T={T} tokens, p={p} dims")

# Flatten to (N, p) for covariance computation
H_flat = H_np.reshape(L * T, p)
H_centered = H_flat - H_flat.mean(axis=0)

# Compute covariance (or a proxy — full p x p may be huge)
# We'll look at the eigenvalue spectrum via SVD
U, S, Vt = np.linalg.svd(H_centered, full_matrices=False)
eigvals_raw = (S ** 2) / (L * T - 1)

print(f"\nCovariance eigenvalue spectrum (raw):")
print(f"  Largest eigenvalue:  {eigvals_raw[0]:.2f}")
print(f"  Smallest eigenvalue: {eigvals_raw[-1]:.6f}")
print(f"  Ratio (max/min):     {eigvals_raw[0] / (eigvals_raw[-1] + 1e-12):.0f}x")
print(f"  Top 10 eigenvalues:  {eigvals_raw[:10].round(2)}")

# Plot diagonal-equivalent: eigenvalue spectrum
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(eigvals_raw[:min(500, len(eigvals_raw))], color='#EE6677', linewidth=1.5)
ax.axhline(y=1.0, color='grey', linestyle='--', alpha=0.5, label='Target (identity): eigenvalue = 1')
ax.set_xlabel('Principal component index')
ax.set_ylabel('Eigenvalue (log scale)')
ax.set_title('Raw Hidden-State Covariance Spectrum — Far From Identity')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.savefig('ch04_raw_eigenvalue_spectrum.png', dpi=150, bbox_inches='tight')
print("\nSaved: ch04_raw_eigenvalue_spectrum.png")
plt.show()

print("\n=> The covariance is wildly anisotropic. Some dimensions have")
print("   variance 10,000x larger than others. This is NOT an orthonormal frame.")

## What Goes Wrong

Without whitening, every geometric quantity is contaminated by the distorted metric. Here is a concrete demonstration: we take two hidden-state vectors, compute the angle between them in raw coordinates, then compute the angle after whitening. The two answers differ — sometimes dramatically — because the raw dot product is dominated by whichever dimensions happen to have the largest variance.

Think of it this way: if dimension 47 has variance $10{,}000$ and dimension 200 has variance $0.01$, then the raw dot product is almost entirely determined by dimension 47. The contribution of dimension 200 is invisible, even if it carries important information. Whitening rescales all dimensions to have equal variance, so every direction contributes fairly to inner products.

The same distortion affects bivectors. A bivector extracted from raw coordinates will have its principal planes tilted toward the high-variance directions, producing "planes" that are artifacts of the coordinate system rather than genuine features of the transformation.

In [ ]:
# ------------------------------------------------------------------
# Demonstration: angles differ between raw and whitened spaces
# ------------------------------------------------------------------

# Pick two hidden states: same token at two different layers
layer_a, layer_b = 5, 20
token_idx = 0  # first token

v_raw_a = H_np[layer_a, token_idx, :]   # raw hidden state at layer 5
v_raw_b = H_np[layer_b, token_idx, :]   # raw hidden state at layer 20

# Angle in raw coordinates
cos_raw = np.dot(v_raw_a, v_raw_b) / (
    np.linalg.norm(v_raw_a) * np.linalg.norm(v_raw_b) + 1e-12
)
angle_raw = np.arccos(np.clip(cos_raw, -1, 1))

# Now whiten
k = 256
metric = core.estimate_metric(H_flat, n_components=min(k, min(L * T, p) - 1))
H_whitened = core.whiten(H_np, metric)

v_white_a = H_whitened[layer_a, token_idx, :]
v_white_b = H_whitened[layer_b, token_idx, :]

# Angle in whitened coordinates
cos_white = np.dot(v_white_a, v_white_b) / (
    np.linalg.norm(v_white_a) * np.linalg.norm(v_white_b) + 1e-12
)
angle_white = np.arccos(np.clip(cos_white, -1, 1))

print(f"Comparing hidden states: layer {layer_a} vs layer {layer_b}, token {token_idx}")
print(f"{'':>25} {'Raw':>12} {'Whitened':>12}")
print(f"{'─' * 50}")
print(f"  {'Cosine similarity':>23}: {cos_raw:>12.4f} {cos_white:>12.4f}")
print(f"  {'Angle (radians)':>23}: {angle_raw:>12.4f} {angle_white:>12.4f}")
print(f"  {'Angle (degrees)':>23}: {np.degrees(angle_raw):>12.1f} {np.degrees(angle_white):>12.1f}")
print(f"\n  Difference: {np.degrees(abs(angle_raw - angle_white)):.1f} degrees!")
print("\n=> The raw and whitened angles disagree. The raw angle is distorted")
print("   by the anisotropic metric. Only the whitened angle is geometrically valid.")

## Whitening = Frame Construction

Whitening is the process of constructing an orthonormal frame from the data. It has three steps:

1. **Center**: subtract the mean $\bar{H}$ so the data is zero-mean.
2. **Diagonalize**: compute the SVD (or eigendecomposition) of the centered data. The right singular vectors $V$ give the principal directions; the singular values $S$ give the scale along each direction.
3. **Scale**: divide each direction by $\sqrt{\lambda_j}$ (the square root of the corresponding eigenvalue). This makes the variance along every direction equal to 1.

The result is the **whitening matrix** $W = V_k \, \mathrm{diag}(1/\sqrt{\lambda_j})$, and the whitened states are:

$$\tilde{H} = (H - \bar{H}) \, W$$

After whitening, the covariance of $\tilde{H}$ is the **identity matrix** $I_k$. This means $g_{ij} = \delta_{ij}$: we are in an orthonormal frame and GA is valid.

Let us perform whitening and visualize the before/after covariance matrices side by side.

In [ ]:
# ------------------------------------------------------------------
# Whitening: before and after covariance
# ------------------------------------------------------------------

# Whitened states already computed above: H_whitened, shape (L, T, k_actual)
k_actual = H_whitened.shape[2]
H_white_flat = H_whitened.reshape(-1, k_actual)

# Compute covariance of whitened states (should be ~identity)
cov_whitened = np.cov(H_white_flat, rowvar=False)  # (k, k)

# For raw covariance, use top-k eigenvalues as proxy
# (full p x p covariance is too large to visualize meaningfully)
# Instead, project raw states to the same k dims WITHOUT scaling
H_proj_flat = H_centered @ metric.V_k  # projected but NOT whitened
cov_raw_proj = np.cov(H_proj_flat, rowvar=False)  # (k, k)

# ── Side-by-side heatmaps ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Show only the top-left 50x50 corner for visual clarity
show_k = min(50, k_actual)

# Before whitening: projected covariance
im1 = ax1.imshow(cov_raw_proj[:show_k, :show_k], cmap='RdBu_r', aspect='equal',
                  vmin=-np.percentile(np.abs(cov_raw_proj[:show_k, :show_k]), 95),
                  vmax=np.percentile(np.abs(cov_raw_proj[:show_k, :show_k]), 95))
ax1.set_title(f'Before Whitening: Projected Covariance\n(top {show_k}x{show_k} block)',
              fontsize=11)
ax1.set_xlabel('Dimension $j$')
ax1.set_ylabel('Dimension $i$')
plt.colorbar(im1, ax=ax1, shrink=0.8)

# After whitening: should be ~identity
im2 = ax2.imshow(cov_whitened[:show_k, :show_k], cmap='RdBu_r', aspect='equal',
                  vmin=-0.5, vmax=1.5)
ax2.set_title(f'After Whitening: Covariance $\\approx I_k$\n(top {show_k}x{show_k} block)',
              fontsize=11)
ax2.set_xlabel('Dimension $j$')
ax2.set_ylabel('Dimension $i$')
plt.colorbar(im2, ax=ax2, shrink=0.8)

fig.suptitle('Whitening Transforms the Metric to $g_{ij} = \\delta_{ij}$', fontsize=13, y=1.02)
fig.tight_layout()
plt.savefig('ch04_covariance_before_after.png', dpi=150, bbox_inches='tight')
print("Saved: ch04_covariance_before_after.png")
plt.show()

print(f"\nBefore whitening:")
print(f"  Diagonal range: [{np.diag(cov_raw_proj).min():.2f}, {np.diag(cov_raw_proj).max():.2f}]")
print(f"  Mean |off-diagonal|: {np.mean(np.abs(cov_raw_proj - np.diag(np.diag(cov_raw_proj)))):.4f}")
print(f"\nAfter whitening:")
print(f"  Diagonal range: [{np.diag(cov_whitened).min():.4f}, {np.diag(cov_whitened).max():.4f}]")
print(f"  Mean |off-diagonal|: {np.mean(np.abs(cov_whitened - np.diag(np.diag(cov_whitened)))):.6f}")

## The Clifford Algebra $\mathrm{Cl}(k, 0)$

Now that $g_{ij} = \delta_{ij}$, we are working in the Clifford algebra $\mathrm{Cl}(k, 0)$ — the algebra generated by $k$ orthonormal basis vectors with the relation:

$$e_i e_j + e_j e_i = 2\delta_{ij}$$

The signature $(k, 0)$ means all basis vectors square to $+1$ (no negative or zero signatures). This is the standard Euclidean Clifford algebra, and all the GA operations we have been using — wedge products, rotors, bivector decompositions — are valid in this algebra.

The dimension $k$ is our whitening dimension. With $k = 256$, we work in $\mathrm{Cl}(256, 0)$, which has:
- 256 basis vectors (grade 1)
- $\binom{256}{2} = 32{,}640$ basis bivectors (grade 2)
- Up to 128 independent planes of rotation

This is vastly richer than $\mathrm{Cl}(3, 0)$ (the 3D algebra with just 3 rotation planes), and it is exactly the algebraic structure we need to capture what transformers do to their representations.

Let us verify that the whitened covariance truly satisfies the orthonormality condition.

In [ ]:
# ------------------------------------------------------------------
# Verify: whitened covariance ≈ identity
# ------------------------------------------------------------------

# Diagonal elements should be ≈ 1
diag_vals = np.diag(cov_whitened)
mean_diag = np.mean(diag_vals)
std_diag = np.std(diag_vals)

# Off-diagonal elements should be ≈ 0
mask_offdiag = ~np.eye(k_actual, dtype=bool)
offdiag_vals = cov_whitened[mask_offdiag]
mean_abs_offdiag = np.mean(np.abs(offdiag_vals))
max_abs_offdiag = np.max(np.abs(offdiag_vals))

print(f"Whitened covariance verification (k = {k_actual}):")
print(f"  Basis bivectors: C({k_actual}, 2) = {k_actual * (k_actual - 1) // 2:,}")
print(f"  Max rotation planes: {k_actual // 2}")
print()
print(f"  Diagonal (should be 1.0):")
print(f"    Mean:  {mean_diag:.6f}")
print(f"    Std:   {std_diag:.6f}")
print(f"    Range: [{diag_vals.min():.6f}, {diag_vals.max():.6f}]")
print()
print(f"  Off-diagonal (should be 0.0):")
print(f"    Mean |off-diag|: {mean_abs_offdiag:.6f}")
print(f"    Max  |off-diag|: {max_abs_offdiag:.6f}")
print()

# Frobenius distance from identity
dist_from_I = np.linalg.norm(cov_whitened - np.eye(k_actual), 'fro')
print(f"  ||Cov - I||_F = {dist_from_I:.6f}")
print(f"  ||Cov - I||_F / k = {dist_from_I / k_actual:.6f}")
print()

# Variance explained
print(f"  Variance explained by k={k_actual} components: {metric.explained_var:.4f} "
      f"({metric.explained_var * 100:.1f}%)")
print()
print(f"=> g_ij ≈ delta_ij. We are in Cl({k_actual}, 0). GA is valid.")

## Choosing $k$

The whitening dimension $k$ is the number of principal components we retain. It controls a trade-off:

- **Too small** ($k \ll 128$): we lose geometric structure. Important planes of rotation may be discarded.
- **Too large** ($k \to p$): we include noisy, low-variance dimensions. The whitening becomes ill-conditioned (dividing by tiny eigenvalues amplifies noise).
- **Just right** ($k \approx 256$): retains $>99.5\%$ of the total variance while keeping the covariance well-conditioned.

Empirically, the geometric structure (rotor angles, principal planes, holonomy curvature) stabilizes by around $k \approx 128$ and is robust across $k \in [128, 512]$. We use $k = 256$ as the default because it balances fidelity with computational efficiency.

The key diagnostic is the **explained variance**: what fraction of the total data variance is captured by the top $k$ components? If it exceeds 99%, the discarded dimensions contribute negligible geometric information.

## Exercises

1. **Eigenvalue ratio.** Compute the ratio of the largest to smallest eigenvalue of the raw covariance (the *condition number* of the metric). How does this compare to the condition number after whitening (which should be 1.0)? What would happen to a bivector computation if this ratio were $10^6$?

2. **Whitening at different $k$.** Re-run the whitening with $k = 64$, $k = 128$, $k = 256$, and $k = 512$. For each, compute (a) explained variance, (b) mean diagonal of the whitened covariance, and (c) the rotor angle profile across layers. At what $k$ does the angle profile stabilize?

3. **Angle distortion map.** For every pair of adjacent layers $(l, l+1)$, compute the angle between the token-0 hidden states in both raw and whitened coordinates. Plot the raw angles and whitened angles on the same axis. Where is the distortion largest?

4. **Off-diagonal structure.** The off-diagonal elements of the raw projected covariance are not all zero — they show correlations between principal components. What pattern do you see? Are nearby components more correlated than distant ones? What does this mean about the structure of the hidden-state space?

5. **Alternative whitening.** Instead of PCA whitening (which uses the eigenvectors of the covariance), try ZCA whitening: $W_{\mathrm{ZCA}} = V \, \mathrm{diag}(1/\sqrt{\lambda_j}) \, V^T$. This whitens the data while staying as close as possible to the original coordinate system. Compare the rotor angle profiles from PCA and ZCA whitening. Do they agree?